# Reproducing WILLIE-MINI inside PyHealth

This notebook re-implements the **MINI model** of WILLIE (Wound Imaging, Localization, and Integrated Evaluation; arXiv 2610.05341) with PyHealth's `Dataset` / `Task` / `Model` abstractions and checks it against the authors' public weights.

**Result in one line:** loaded with the public fold-0 checkpoint, the PyHealth pipeline reproduces the original code's logits exactly (max difference 0.0, 100% argmax agreement), gives 87.61% accuracy on the 234-image test set, and 84.72% on the public fold-0 validation indices (183/216) — the numbers recorded by the authors.

| Component | File (inside PyHealth) | Purpose |
|---|---|---|
| Processors | `processors/wound_processors.py` | The original cv2 bilinear resize + ImageNet normalisation; nearest-neighbour mask loading |
| Dataset | `datasets/willie.py`, `datasets/configs/willie.yaml` | AZH, Medetec and FUSeg manifests merged into one metadata CSV; public 5-fold indices attached |
| Tasks | `tasks/willie_tasks.py` | `WILLIEClassification` (5 classes), `WILLIESegmentation` (FUSeg masks) |
| Model | `models/willie_mini.py` | `WILLIEMini` — module names are identical to the original, so the public weights load with `strict=True` semantics |
| Tests | `tests/core/test_willie.py` | 19 tests using synthetic images and a tiny stand-in backbone, including an equivalence test against the original code |

**Architecture actually used** (from the checkpoint's saved `config`, which differs from the README's "224×224, MoE-8"): input 518×518, DINOv2 ViT-S/14 backbone, FPN 256, one WA-CSA layer, MoE with **2 experts** (top-k 2), 34.32M parameters.

**Run order:** §1 (setup and downloads) → §2 → §3 → restart the runtime after §3's registration cell → re-run the setup cell of §1 (variables only) → §4 → §5 → §6. A T4 GPU runtime is sufficient.

## 1. Setup and downloads
Run these cells first and in order. Everything downloaded here is stored on Google Drive (`BENCH_DIR`), so a later runtime restart does not require downloading again. If Drive space is short, point `BENCH_DIR` at `/content/willie-benchmark` instead.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

HF_DATASET   = "QianGroup/willie-benchmark"   # data.tar.gz, manifests/, 5fold_splits_v2.pt
HF_WEIGHTS   = "QianGroup/willie-weights"     # public checkpoints (downloaded in section 2)
BENCH_DIR    = Path("/content/drive/MyDrive/willie/willie-benchmark")
REPO_DIR     = Path("/content/Willie")        # GitHub repo: code and manifests
PYHEALTH_DIR = Path("/content/PyHealth")
FOLDS_PT     = BENCH_DIR / "manifests/5fold_splits_v2.pt"          # public 5-fold indices
REF_LOGITS   = BENCH_DIR.parent / "willie_mini_fold0_reference_logits.pt"  # written in section 2
CKPT_DIR     = BENCH_DIR.parent / "pyhealth_runs"
BENCH_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1) HuggingFace: benchmark data (data.tar.gz, manifests, 5-fold indices)
from huggingface_hub import snapshot_download
snapshot_download(repo_id=HF_DATASET, repo_type="dataset", local_dir=str(BENCH_DIR))

In [ ]:
# 2) GitHub: code and manifests, 3) unpack the data into the repo root and verify it
%cd /content
!test -d {REPO_DIR} || git clone https://github.com/Qian-Group-HRI/Willie.git {REPO_DIR}
%cd {REPO_DIR}
!tar -xzf {BENCH_DIR}/data.tar.gz   # must be extracted at the repo root so the relative data/ paths resolve
!python verify_data.py

/content/Willie
directory                 found  missing
----------------------------------------
data/AZH/test/BG             25        0
data/AZH/test/diabetic       46        0
data/AZH/test/no wound       25        0
data/AZH/test/pressure       34        0
data/AZH/test/surgical       42        0
data/AZH/test/venous         62        0
data/AZH/train/BG            75        0
data/AZH/train/diabetic     139        0
data/AZH/train/no wound      75        0
data/AZH/train/pressure     100        0
data/AZH/train/surgical     122        0
data/AZH/train/venous       185        0
data/FUSeg/test/images      200        0
data/FUSeg/train/images     610        0
data/FUSeg/train/labels     610        0
data/FUSeg/val/images       400        0
data/FUSeg/val/labels       400        0
data/Medetec/diabetic        48        0
data/Medetec/pressure       170        0
data/Medetec/toes            34        0
data/Medetec/venous         133        0
----------------------------------------


## 2. Baseline with the original code
The model definition lives in the first cell of the authors' `06_WILLIE_MINI.ipynb`. We execute only its model-definition part (sections 2–11; no cluster paths or manifest loading) to obtain the reference implementation. Running the public fold-0 weights on `cls_test` gives the baseline, and the saved logits become the ground truth for the PyHealth comparison in §5.

In [ ]:
!pip -q install einops
import json, torch

def load_willie_definitions(nb_path):
    """Execute only the model-definition part (# 2 ... # 11) of 06_WILLIE_MINI.ipynb."""
    src = "".join(json.load(open(nb_path))["cells"][0]["source"])
    body = src[src.index("# 2. MODEL CONFIGURATIONS"): src.index("# 12. CHECKPOINT MANAGER")]
    prelude = (
        "import torch, torch.nn as nn, torch.nn.functional as F\n"
        "from dataclasses import dataclass, field\n"
        "from typing import List, Optional, Dict, Tuple, Any\n"
        "from einops import rearrange\n"
    )
    ns = {}
    exec(prelude + body, ns)
    return ns

ns = load_willie_definitions(REPO_DIR / "06_WILLIE_MINI.ipynb")
cfg = ns["get_model_config"]("MINI")
model = ns["WILLIEModel"](cfg)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M  (README: 34.3M)")

params: 34.32M  (README: 34.3M)


In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(HF_WEIGHTS, "mini/willie_mini_fold0_best.pt")
# Official checkpoint from the authors' repository, so weights_only=False
# (weights_only=True fails because the file contains numpy scalars). Do not do this for untrusted files.
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(ckpt.keys())
print("saved config:", ckpt.get("config"))
res = model.load_state_dict(ckpt["model_state_dict"], strict=False)
print("missing:", res.missing_keys[:10], "| unexpected:", res.unexpected_keys[:10])

dict_keys(['model_state_dict', 'epoch', 'fold', 'metrics', 'config', 'optimizer_state_dict', 'scheduler_state_dict'])
saved config: {'name': 'MINI', 'backbone_name': 'dinov2_vits14', 'backbone_dim': 384, 'backbone_layers': [2, 5, 8, 11], 'img_size': 518, 'patch_size': 14, 'fpn_dim': 256, 'fpn_levels': 4, 'wa_csa_layers': 1, 'wa_csa_heads': 8, 'wa_csa_dropout': 0.1, 'num_classes': 5, 'num_experts': 2, 'top_k_experts': 2, 'cls_embed_dim': 128, 'seg_out_channels': 1, 'pscse_reduction': 16, 'det_max_objects': 20, 'freeze_backbone_epochs': 3}
missing: [] | unexpected: []


In [ ]:
import cv2, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, f1_score

DEV  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_img(path, size=518):
    img = cv2.cvtColor(cv2.imread(str(REPO_DIR / path)), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)   # same as albumentations A.Resize
    img = (img.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(img).permute(2, 0, 1)

print({k: (round(v, 2) if isinstance(v, float) else v)
       for k, v in ckpt["metrics"].items()
       if k in ("epoch", "cls_acc", "cls_f1", "seg_dice", "det_ap50", "combined_weighted")})

model = model.to(DEV).eval()
test = pd.read_csv(REPO_DIR / "Manifests_and_Splits/manifests/cls_test.csv")
paths, labels = test["image_path"].tolist(), test["unified_label"].astype(int).tolist()
all_logits = []
with torch.no_grad():
    for i in range(0, len(paths), 8):
        x = torch.stack([load_img(p) for p in paths[i:i + 8]]).to(DEV)
        all_logits.append(model(x, target_seg_size=512)["cls_logits"].float().cpu())
logits_ref = torch.cat(all_logits)
preds = logits_ref.argmax(1).tolist()
print(f"cls_test n={len(labels)}  acc={accuracy_score(labels, preds)*100:.2f}%  "
      f"macroF1={f1_score(labels, preds, average='macro')*100:.2f}%")
torch.save({"paths": paths, "labels": labels, "logits": logits_ref}, REF_LOGITS)   # ground truth for section 5

{'epoch': 9, 'cls_acc': 84.72, 'cls_f1': 84.75, 'seg_dice': 79.18, 'det_ap50': 81.66, 'combined_weighted': 81.89}
cls_test n=234  acc=87.61%  macroF1=85.54%


**Reading the result.** 87.61% on the 234-image test set is consistent with the README's 5-fold MINI average (86.4 ± 1.2). With n = 234 the 95% confidence interval is roughly ±4 percentage points, so differences of 1–2 points are not meaningful. The validation metrics stored in the checkpoint (Acc 84.72 / Dice 79.18 / AP 81.66) are the authors' own training-log values for fold 0; the README headline numbers (e.g. 91.88%) come from a 5-fold ensemble with test-time augmentation and are not comparable to a single fold.

## 3. PyHealth implementation
Clone and install PyHealth, write the new files, and register them. The registration appends **one line per `__init__.py`** and never overwrites an existing file (overwriting breaks unrelated imports).
Before submission, replace the `<your name> (<your NetID>)` placeholder in each file header.

In [ ]:
%cd /content
!test -d {PYHEALTH_DIR} || git clone https://github.com/sunlabuiuc/PyHealth.git {PYHEALTH_DIR}
%cd {PYHEALTH_DIR}
!pip install -e . -q

In [ ]:
%%writefile /content/PyHealth/pyhealth/processors/wound_processors.py
"""Image and mask processors for the WILLIE wound benchmark.

``WoundImageProcessor`` reproduces the original WILLIE preprocessing exactly
(``cv2.resize`` with bilinear interpolation, scaling to [0, 1], ImageNet
normalisation). It is not the same as :class:`ImageProcessor`, whose
torchvision ``Resize`` anti-aliases when it shrinks an image, so the two give
slightly different pixels and therefore slightly different logits.

``WoundMaskProcessor`` reads a binary wound mask and resizes it with
nearest-neighbour interpolation so that no intermediate grey values appear.

Author: <your name> (<your NetID>)
Paper: WILLIE, arXiv 2610.05341
"""

from pathlib import Path
from typing import Any, List, Optional, Union

import numpy as np
import torch

from . import register_processor
from .base_processor import FeatureProcessor

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _cv2():
    try:
        import cv2
    except ImportError as exc:  # pragma: no cover
        raise ImportError(
            "The WILLIE processors need OpenCV: pip install opencv-python-headless"
        ) from exc
    return cv2


@register_processor("wound_image")
class WoundImageProcessor(FeatureProcessor):
    """Loads an RGB image, resizes it (bilinear, cv2) and normalises it.

    Args:
        image_size: Side of the square output. Defaults to 518 (WILLIE-MINI).
        mean: Per-channel mean. Defaults to the ImageNet mean.
        std: Per-channel standard deviation. Defaults to the ImageNet std.
    """

    stores_tensor = True

    def __init__(
        self,
        image_size: int = 518,
        mean: Optional[List[float]] = None,
        std: Optional[List[float]] = None,
    ) -> None:
        self.image_size = image_size
        self.mean = np.asarray(mean if mean is not None else IMAGENET_MEAN, np.float32)
        self.std = np.asarray(std if std is not None else IMAGENET_STD, np.float32)

    def process(self, value: Union[str, Path]) -> torch.Tensor:
        """Returns a ``(3, image_size, image_size)`` float tensor."""
        cv2 = _cv2()
        path = Path(value)
        if not path.exists():
            raise FileNotFoundError(f"Image file not found: {path}")
        img = cv2.imread(str(path))
        if img is None:
            raise ValueError(f"Could not decode image: {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(
            img, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR
        )
        img = (img.astype(np.float32) / 255.0 - self.mean) / self.std
        return torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))

    def is_token(self) -> bool:
        return False

    def schema(self) -> tuple:
        return ("value",)

    def dim(self) -> tuple:
        return (3,)

    def spatial_dim(self) -> tuple:
        return (2,)


@register_processor("wound_mask")
class WoundMaskProcessor(FeatureProcessor):
    """Loads a wound mask as a binary ``(1, size, size)`` float tensor.

    Args:
        mask_size: Side of the square output. Defaults to 512, the
            segmentation resolution of WILLIE-MINI.
        threshold: Pixels above this grey value (0-255) are foreground.
            Defaults to 127, as in the original data loader.
    """

    stores_tensor = True

    def __init__(self, mask_size: int = 512, threshold: int = 127) -> None:
        self.mask_size = mask_size
        self.threshold = threshold

    def process(self, value: Any) -> torch.Tensor:
        cv2 = _cv2()
        path = Path(value)
        if not path.exists():
            raise FileNotFoundError(f"Mask file not found: {path}")
        mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not decode mask: {path}")
        mask = (mask > self.threshold).astype(np.uint8)
        mask = cv2.resize(
            mask, (self.mask_size, self.mask_size), interpolation=cv2.INTER_NEAREST
        )
        return torch.from_numpy(mask.astype(np.float32))[None]

    def is_token(self) -> bool:
        return False

    def schema(self) -> tuple:
        return ("value",)

    def dim(self) -> tuple:
        return (3,)

    def spatial_dim(self) -> tuple:
        return (2,)

In [ ]:
%%writefile /content/PyHealth/pyhealth/datasets/configs/willie.yaml
version: "1.0"
tables:
  willie:
    file_path: "willie-metadata-pyhealth.csv"
    patient_id: null
    timestamp: null
    attributes:
    - "image"
    - "mask"
    - "label"
    - "label_name"
    - "source"
    - "split"
    - "cv_fold"
    - "kind"

In [ ]:
%%writefile /content/PyHealth/pyhealth/tasks/willie_tasks.py
"""Tasks for the WILLIE wound benchmark.

Paper: WILLIE, arXiv 2610.05341. Dataset: :class:`~pyhealth.datasets.WILLIEDataset`.

Author: <your name> (<your NetID>)
"""

from typing import Any, Dict, List, Optional

from .base_task import BaseTask


class _WILLIETask(BaseTask):
    """Shared filtering by split and cross-validation fold."""

    event_kind: str = ""

    def __init__(
        self,
        split: Optional[str] = None,
        fold: Optional[int] = None,
        fold_role: Optional[str] = None,
        image_size: int = 518,
    ) -> None:
        """
        Args:
            split: Keep only this benchmark split (``"train"``, ``"val"``,
                ``"test"``). ``None`` keeps all labelled rows of the task.
            fold: With ``fold_role``, use the published cross-validation
                fold: rows of train+val whose ``cv_fold`` equals ``fold`` are
                ``"val"``, all other train+val rows are ``"train"``.
            fold_role: ``"train"`` or ``"val"``. Needs the dataset to be
                built with ``folds_path``.
            image_size: Side of the processed image (518 for WILLIE-MINI).
        """
        if fold_role not in (None, "train", "val"):
            raise ValueError("fold_role must be None, 'train' or 'val'.")
        if (fold is None) != (fold_role is None):
            raise ValueError("fold and fold_role must be given together.")
        if fold_role is not None and split is not None:
            raise ValueError("Use either split or fold/fold_role, not both.")
        self.split = split
        self.fold = fold
        self.fold_role = fold_role
        self.image_size = image_size
        self.input_schema = {"image": ("wound_image", {"image_size": image_size})}
        super().__init__()

    def _keep(self, event) -> bool:
        if event["kind"] != self.event_kind:
            return False
        if self.fold_role is not None:
            if event["split"] == "test":
                return False
            in_val = int(event["cv_fold"]) == self.fold
            return in_val if self.fold_role == "val" else not in_val
        return self.split is None or event["split"] == self.split


class WILLIEClassification(_WILLIETask):
    """Five-class wound classification (diabetic, pressure, surgical, venous,
    no wound) on AZH and Medetec images.

    The label is the integer ``unified_label`` of the benchmark, so class
    indices match the released WILLIE weights.

    Examples:
        >>> task = WILLIEClassification(split="test")
        >>> task.task_name
        'WILLIEClassification'
    """

    task_name: str = "WILLIEClassification"
    event_kind = "cls"

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.output_schema = {"label": ("multiclass", {"labels": 5})}

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        samples = []
        for event in patient.get_events(event_type="willie"):
            if self._keep(event):
                samples.append(
                    {
                        "patient_id": patient.patient_id,
                        "image": event["image"],
                        "image_path": event["image"],
                        "label": int(event["label"]),
                        "source": event["source"],
                    }
                )
        return samples


class WILLIESegmentation(_WILLIETask):
    """Binary wound segmentation on FUSeg images.

    Samples hold the processed image and a ``(1, mask_size, mask_size)``
    binary mask. FUSeg test images have no public mask and are not included.

    Args:
        mask_size: Side of the processed mask (512, the WILLIE-MINI output).
    """

    task_name: str = "WILLIESegmentation"
    event_kind = "seg"

    def __init__(self, *args, mask_size: int = 512, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.mask_size = mask_size
        self.output_schema = {"mask": ("wound_mask", {"mask_size": mask_size})}

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        samples = []
        for event in patient.get_events(event_type="willie"):
            if self._keep(event):
                samples.append(
                    {
                        "patient_id": patient.patient_id,
                        "image": event["image"],
                        "image_path": event["image"],
                        "mask": event["mask"],
                        "source": event["source"],
                    }
                )
        return samples

In [ ]:
%%writefile /content/PyHealth/pyhealth/datasets/willie.py
"""WILLIE wound benchmark dataset for PyHealth.

Dataset: https://huggingface.co/datasets/QianGroup/willie-benchmark
Code and manifests: https://github.com/Qian-Group-HRI/Willie
Paper: WILLIE (Wound Imaging, Localization, and Integrated Evaluation),
    arXiv 2610.05341.

The benchmark merges three public wound datasets (AZH, Medetec, FUSeg). The
classification images (AZH + Medetec) carry a five-class label, the FUSeg
images carry a binary wound mask.

Author: <your name> (<your NetID>)
"""

import logging
import os
from pathlib import Path
from typing import Optional

import pandas as pd

from ..tasks import WILLIEClassification
from .base_dataset import BaseDataset

logger = logging.getLogger(__name__)

# Five-class label space the released WILLIE weights were trained on
# (``unified_label`` of the classification manifests).
WILLIE_CLASSES = ["diabetic", "pressure", "surgical", "venous", "no_wound"]


class WILLIEDataset(BaseDataset):
    """Wound classification and segmentation benchmark (AZH, Medetec, FUSeg).

    ``prepare_metadata`` writes ``willie-metadata-pyhealth.csv`` into ``root``
    with one row per image (absolute paths). Rows have ``kind="cls"``
    (classification, from ``Manifests_and_Splits/manifests/cls_*.csv``) or
    ``kind="seg"`` (FUSeg segmentation, from
    ``data/processed/index/data_index.csv``). Use
    :class:`~pyhealth.tasks.WILLIEClassification` or
    :class:`~pyhealth.tasks.WILLIESegmentation` to build samples.

    Splits: ``split`` is the benchmark's ``train``/``val``/``test`` column. For
    the published 5-fold cross-validation, pass ``folds_path`` (the
    ``5fold_splits_v2.pt`` file shipped with the HF dataset); ``cv_fold`` then
    holds the fold in which a train+val row is the validation row (``-1`` for
    test rows and when no folds file is given). The fold indices refer to
    ``concat(cls_train, cls_val)`` (classification) and to the FUSeg
    train+val rows in index order (segmentation); the segmentation order is
    not verified against the original manifests.

    Args:
        root: Root of the GitHub repo checkout, with ``data/`` unpacked from
            ``data.tar.gz`` (so that ``verify_data.py`` passes).
        folds_path: Optional path to ``5fold_splits_v2.pt``.
        dataset_name: Optional name. Defaults to ``"willie"``.
        config_path: Optional yaml; defaults to ``configs/willie.yaml``.
        cache_dir: Optional cache directory.
        num_workers: Parallel workers.
        dev: Dev mode (limits the number of patients).

    Examples:
        >>> from pyhealth.datasets import WILLIEDataset
        >>> from pyhealth.tasks import WILLIEClassification
        >>> dataset = WILLIEDataset(root="/content/Willie")
        >>> samples = dataset.set_task(WILLIEClassification(split="test"))
    """

    def __init__(
        self,
        root: str,
        folds_path: Optional[str] = None,
        dataset_name: Optional[str] = None,
        config_path: Optional[str] = None,
        cache_dir: Optional[str] = None,
        num_workers: int = 1,
        dev: bool = False,
    ) -> None:
        if config_path is None:
            logger.info("No config path provided, using default config")
            config_path = Path(__file__).parent / "configs" / "willie.yaml"
        self.prepare_metadata(root, folds_path)
        super().__init__(
            root=root,
            tables=["willie"],
            dataset_name=dataset_name or "willie",
            config_path=config_path,
            cache_dir=cache_dir,
            num_workers=num_workers,
            dev=dev,
        )

    @staticmethod
    def prepare_metadata(root: str, folds_path: Optional[str] = None) -> pd.DataFrame:
        """Builds ``willie-metadata-pyhealth.csv`` in ``root``.

        Args:
            root: Repo root (contains ``Manifests_and_Splits`` and ``data``).
            folds_path: Optional ``5fold_splits_v2.pt``.

        Returns:
            The metadata table that was written.

        Raises:
            FileNotFoundError: If a manifest or an image is missing.
        """
        manifests = os.path.join(root, "Manifests_and_Splits", "manifests")
        parts = {
            s: pd.read_csv(os.path.join(manifests, f"cls_{s}.csv"))
            for s in ("train", "val", "test")
        }
        trainval = pd.concat([parts["train"], parts["val"]], ignore_index=True)
        trainval["split"] = ["train"] * len(parts["train"]) + ["val"] * len(parts["val"])
        trainval["cv_fold"] = -1
        test = parts["test"].assign(split="test", cv_fold=-1)

        folds = None
        if folds_path is not None:
            import torch

            folds = torch.load(folds_path, map_location="cpu", weights_only=True)["folds"]
            for fold, idx in folds.items():
                if max(idx["cls_va"] + idx["cls_tr"]) >= len(trainval):
                    raise ValueError(
                        "folds_path does not match the classification manifests: "
                        f"fold {fold} indexes row {max(idx['cls_va'] + idx['cls_tr'])} "
                        f"but cls_train + cls_val has {len(trainval)} rows."
                    )
                trainval.loc[idx["cls_va"], "cv_fold"] = int(fold)

        cls = pd.concat([trainval, test], ignore_index=True)
        cls_rows = pd.DataFrame(
            {
                "image": cls["image_path"],
                "mask": "",
                "label": cls["unified_label"].astype(int),
                "label_name": cls["unified_class"],
                "source": cls["source"],
                "split": cls["split"],
                "cv_fold": cls["cv_fold"].astype(int),
                "kind": "cls",
            }
        )

        index = pd.read_csv(os.path.join(root, "data", "processed", "index", "data_index.csv"))
        seg = index[(index["task"] == "seg") & (index["is_labeled"] == 1)].reset_index(drop=True)
        seg_cv = pd.Series(-1, index=seg.index)
        if folds is not None:
            trainval_seg = seg[seg["split"].isin(["train", "val"])]
            for fold, idx in folds.items():
                if idx["seg_va"] and max(idx["seg_va"]) < len(trainval_seg):
                    seg_cv.loc[trainval_seg.index[idx["seg_va"]]] = int(fold)
        seg_rows = pd.DataFrame(
            {
                "image": seg["image_path"],
                "mask": seg["mask_path"],
                "label": -1,
                "label_name": "",
                "source": seg["dataset"],
                "split": seg["split"],
                "cv_fold": seg_cv.astype(int),
                "kind": "seg",
            }
        )

        df = pd.concat([cls_rows, seg_rows], ignore_index=True)
        for col in ("image", "mask"):
            df[col] = df[col].map(
                lambda p: os.path.abspath(os.path.join(root, p)) if isinstance(p, str) and p else ""
            )
        missing = [p for p in df["image"] if not os.path.isfile(p)]
        missing += [p for p in df["mask"] if p and not os.path.isfile(p)]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} files are missing (first: {missing[0]}). "
                "Unpack data.tar.gz in the repo root and run verify_data.py."
            )
        df.to_csv(os.path.join(root, "willie-metadata-pyhealth.csv"), index=False)
        return df

    @property
    def default_task(self) -> WILLIEClassification:
        """The default task: five-class wound classification."""
        return WILLIEClassification()

In [ ]:
%%writefile /content/PyHealth/pyhealth/models/willie_mini.py
"""WILLIE-MINI: unified wound classification / segmentation / localization.

Reproduction of the MINI variant of WILLIE (MLHC 2026) for PyHealth.

Paper: WILLIE (Wound Imaging, Localization, and Integrated Evaluation),
    arXiv 2610.05341.
Original code: https://github.com/Qian-Group-HRI/Willie (notebook
    ``06_WILLIE_MINI.ipynb``).
Weights: https://huggingface.co/QianGroup/willie-weights

The module and attribute names (``encoder``, ``fpn``, ``wa_csa``,
``cls_decoder``, ``seg_decoder``, ``det_decoder``) deliberately match the
original implementation, so the published checkpoints load with
:meth:`WILLIEMini.load_original_checkpoint` and the outputs can be compared
tensor by tensor.

Author: <your name> (<your NetID>)
"""

from __future__ import annotations

from dataclasses import dataclass, field, replace
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

from .base_model import BaseModel

# ──────────────────────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────────────────────


@dataclass
class WILLIEConfig:
    """Hyper-parameters of WILLIE-MINI (defaults match the released weights).

    The defaults are the values stored in the published MINI checkpoint
    (``img_size=518``, 2 experts, top-k 2), not the values quoted in the README.
    """

    name: str = "MINI"
    backbone_name: str = "dinov2_vits14"
    backbone_dim: int = 384
    backbone_layers: List[int] = field(default_factory=lambda: [2, 5, 8, 11])
    img_size: int = 518
    patch_size: int = 14
    fpn_dim: int = 256
    fpn_levels: int = 4
    wa_csa_layers: int = 1
    wa_csa_heads: int = 8
    wa_csa_dropout: float = 0.1
    num_classes: int = 5
    num_experts: int = 2
    top_k_experts: int = 2
    cls_embed_dim: int = 128
    seg_out_channels: int = 1
    pscse_reduction: int = 16

    @property
    def grid_size(self) -> int:
        """Side length of the DINOv2 patch grid."""
        return self.img_size // self.patch_size


# ──────────────────────────────────────────────────────────────────────────────
# Building blocks (names and structure follow the original implementation)
# ──────────────────────────────────────────────────────────────────────────────


class DINOv2MultiScale(nn.Module):
    """DINOv2 backbone with forward hooks that expose four intermediate maps.

    Args:
        cfg: Model configuration.
        pretrained: Download the DINOv2 pre-trained weights through
            ``torch.hub``. Set to ``False`` when the weights come from a
            WILLIE checkpoint (which contains the backbone) or in tests.
        backbone: Optional ready-made backbone (anything exposing ``blocks``
            and returning tokens), used instead of ``torch.hub``.
    """

    def __init__(
        self,
        cfg: WILLIEConfig,
        pretrained: bool = True,
        backbone: Optional[nn.Module] = None,
    ):
        super().__init__()
        self.cfg = cfg
        self.features: Dict[int, torch.Tensor] = {}
        if backbone is None:
            backbone = torch.hub.load(
                "facebookresearch/dinov2", cfg.backbone_name, pretrained=pretrained
            )
        self.backbone = backbone
        self._hook_handles = []
        for idx in cfg.backbone_layers:
            handle = self.backbone.blocks[idx].register_forward_hook(
                self._make_hook(idx)
            )
            self._hook_handles.append(handle)
        self.set_frozen(True)

    def _make_hook(self, idx: int):
        def hook(module, inputs, output):
            tokens = output[:, 1:, :]
            grid = self.cfg.grid_size
            self.features[idx] = rearrange(
                tokens, "b (h w) d -> b d h w", h=grid, w=grid
            )

        return hook

    def set_frozen(self, frozen: bool) -> None:
        """Freezes (or unfreezes) every backbone parameter."""
        for param in self.backbone.parameters():
            param.requires_grad = not frozen

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        self.features.clear()
        if x.shape[-1] != self.cfg.img_size:
            x = F.interpolate(
                x, size=self.cfg.img_size, mode="bilinear", align_corners=False
            )
        self.backbone(x)
        return [self.features[idx] for idx in self.cfg.backbone_layers]


class FeaturePyramidNeck(nn.Module):
    """Top-down feature pyramid over the four backbone maps."""

    def __init__(self, cfg: WILLIEConfig):
        super().__init__()
        in_dim, dim = cfg.backbone_dim, cfg.fpn_dim
        self.laterals = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(in_dim, dim, 1, bias=False),
                    nn.GroupNorm(32, dim),
                    nn.GELU(),
                )
                for _ in range(cfg.fpn_levels)
            ]
        )
        self.smooth = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(dim, dim, 3, padding=1, bias=False),
                    nn.GroupNorm(32, dim),
                    nn.GELU(),
                )
                for _ in range(cfg.fpn_levels)
            ]
        )

    def forward(self, features: List[torch.Tensor]) -> List[torch.Tensor]:
        lats = [self.laterals[i](features[i]) for i in range(len(features))]
        pyr: List[Optional[torch.Tensor]] = [None] * len(features)
        pyr[-1] = lats[-1]
        for i in range(len(features) - 2, -1, -1):
            up = F.interpolate(
                pyr[i + 1],
                size=lats[i].shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            pyr[i] = lats[i] + up
        return [self.smooth[i](pyr[i]) for i in range(len(features))]


class WoundAwareCrossScaleAttention(nn.Module):
    """Bidirectional cross-attention between two pyramid levels (WA-CSA)."""

    def __init__(self, cfg: WILLIEConfig, level_idx: int = 0):
        super().__init__()
        dim = cfg.fpn_dim
        self.heads = cfg.wa_csa_heads
        self.head_dim = dim // self.heads
        self.scale = self.head_dim**-0.5
        self.use_sdpa = level_idx > 0
        self.dropout_p = cfg.wa_csa_dropout

        self.q_f = nn.Linear(dim, dim, bias=False)
        self.k_c = nn.Linear(dim, dim, bias=False)
        self.v_c = nn.Linear(dim, dim, bias=False)
        self.q_c = nn.Linear(dim, dim, bias=False)
        self.k_f = nn.Linear(dim, dim, bias=False)
        self.v_f = nn.Linear(dim, dim, bias=False)
        self.out_f = nn.Linear(dim, dim, bias=False)
        self.out_c = nn.Linear(dim, dim, bias=False)

        self.gate_f = nn.Sequential(
            nn.Conv2d(dim, dim // 4, 1),
            nn.GELU(),
            nn.Conv2d(dim // 4, 1, 1),
            nn.Sigmoid(),
        )
        self.gate_c = nn.Sequential(
            nn.Conv2d(dim, dim // 4, 1),
            nn.GELU(),
            nn.Conv2d(dim // 4, 1, 1),
            nn.Sigmoid(),
        )

        self.alpha_f = nn.Parameter(torch.zeros(1))
        self.alpha_c = nn.Parameter(torch.zeros(1))
        self.norm_f = nn.LayerNorm(dim)
        self.norm_c = nn.LayerNorm(dim)
        self.drop = nn.Dropout(cfg.wa_csa_dropout)

    def _attn(self, q, k, v):
        q = rearrange(q, "b n (h d) -> b h n d", h=self.heads)
        k = rearrange(k, "b n (h d) -> b h n d", h=self.heads)
        v = rearrange(v, "b n (h d) -> b h n d", h=self.heads)
        if self.use_sdpa and hasattr(F, "scaled_dot_product_attention"):
            out = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.dropout_p if self.training else 0.0
            )
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            out = self.drop(attn.softmax(-1)) @ v
        return rearrange(out, "b h n d -> b n (h d)")

    def forward(self, fine: torch.Tensor, coarse: torch.Tensor):
        _, _, height, _ = fine.shape
        f_seq = self.norm_f(rearrange(fine, "b c h w -> b (h w) c"))
        c_seq = self.norm_c(rearrange(coarse, "b c h w -> b (h w) c"))

        f_up = rearrange(
            self.out_f(self._attn(self.q_f(f_seq), self.k_c(c_seq), self.v_c(c_seq))),
            "b (h w) c -> b c h w",
            h=height,
        )
        c_up = rearrange(
            self.out_c(self._attn(self.q_c(c_seq), self.k_f(f_seq), self.v_f(f_seq))),
            "b (h w) c -> b c h w",
            h=height,
        )
        return (
            fine + torch.tanh(self.alpha_f) * f_up * self.gate_f(fine),
            coarse + torch.tanh(self.alpha_c) * c_up * self.gate_c(coarse),
        )


class WA_CSA_Stack(nn.Module):
    """Stack of WA-CSA layers applied to neighbouring pyramid levels."""

    def __init__(self, cfg: WILLIEConfig):
        super().__init__()
        pairs = cfg.fpn_levels - 1
        self.layers = nn.ModuleList(
            [
                nn.ModuleList(
                    [WoundAwareCrossScaleAttention(cfg, p) for p in range(pairs)]
                )
                for _ in range(cfg.wa_csa_layers)
            ]
        )

    def forward(self, pyr: List[torch.Tensor]) -> List[torch.Tensor]:
        for layer in self.layers:
            level = list(pyr)
            for i, attn in enumerate(layer):
                level[i], level[i + 1] = attn(level[i], level[i + 1])
            pyr = level
        return pyr


class Expert(nn.Module):
    """One MLP expert of the classification mixture."""

    def __init__(self, d_in: int, d_hid: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hid, d_out),
        )

    def forward(self, x):
        return self.net(x)


class TopKRouter(nn.Module):
    """Top-k softmax router with a load-balancing auxiliary loss."""

    def __init__(self, d_in: int, n_exp: int, top_k: int = 2):
        super().__init__()
        self.n_exp, self.top_k = n_exp, top_k
        self.gate = nn.Linear(d_in, n_exp, bias=False)
        self.aux_loss = 0.0

    def forward(self, x):
        probs = F.softmax(self.gate(x), -1)
        weights, idx = torch.topk(probs, self.top_k, -1)
        weights = weights / weights.sum(-1, keepdim=True)
        self.aux_loss = F.mse_loss(
            probs.mean(0), torch.ones(self.n_exp, device=x.device) / self.n_exp
        )
        return weights, idx


class ClassificationDecoder(nn.Module):
    """Multi-scale pooling, mixture of experts, logits and the wound embedding."""

    def __init__(self, cfg: WILLIEConfig):
        super().__init__()
        dim = cfg.fpn_dim
        self.cfg = cfg
        self.pools = nn.ModuleList(
            [
                nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(1))
                for _ in range(cfg.fpn_levels)
            ]
        )
        self.scale_attn = nn.Sequential(
            nn.Linear(cfg.fpn_levels, cfg.fpn_levels), nn.Softmax(-1)
        )
        self.pre = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU())
        self.experts = nn.ModuleList(
            [Expert(dim, dim * 2, cfg.num_classes) for _ in range(cfg.num_experts)]
        )
        self.router = TopKRouter(dim, cfg.num_experts, cfg.top_k_experts)
        self.embed_head = nn.Sequential(
            nn.Linear(dim, cfg.cls_embed_dim),
            nn.LayerNorm(cfg.cls_embed_dim),
            nn.GELU(),
        )

    def forward(self, pyr: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        batch = pyr[0].shape[0]
        pooled = torch.stack(
            [self.pools[i](pyr[i]) for i in range(self.cfg.fpn_levels)], 1
        )
        scale_w = self.scale_attn(
            torch.ones(batch, self.cfg.fpn_levels, device=pooled.device)
        )
        fused = self.pre((pooled * scale_w.unsqueeze(-1)).sum(1))
        weights, idx = self.router(fused)
        logits = torch.zeros(batch, self.cfg.num_classes, device=fused.device)
        for k in range(self.cfg.top_k_experts):
            for e in range(self.cfg.num_experts):
                mask = idx[:, k] == e
                if mask.any():
                    logits[mask] += weights[mask, k : k + 1] * self.experts[e](
                        fused[mask]
                    )
        return logits, self.embed_head(fused)


class ChannelSE(nn.Module):
    """Squeeze-and-excitation over channels."""

    def __init__(self, ch: int, r: int = 16):
        super().__init__()
        mid = max(ch // r, 4)
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(1),
            nn.Linear(ch, mid, bias=False),
            nn.ReLU(True),
            nn.Linear(mid, ch, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x).view(x.shape[0], -1, 1, 1)


class SpatialSE(nn.Module):
    """Squeeze-and-excitation over space."""

    def __init__(self, ch: int):
        super().__init__()
        self.conv = nn.Conv2d(ch, 1, 1, bias=False)

    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))


class ParallelScSE(nn.Module):
    """P-scSE of FUSegNet: additive + maxout of channel and spatial SE."""

    def __init__(self, ch: int, r: int = 16, shortened: bool = False):
        super().__init__()
        self.shortened = shortened
        self.cse_a, self.sse_a = ChannelSE(ch, r), SpatialSE(ch)
        if not shortened:
            self.cse_m, self.sse_m = ChannelSE(ch, r), SpatialSE(ch)

    def forward(self, x):
        add = self.cse_a(x) + self.sse_a(x)
        if self.shortened:
            return add
        return add + torch.max(self.cse_m(x), self.sse_m(x))


class FiLMConditioner(nn.Module):
    """Feature-wise modulation of segmentation features by the wound embedding."""

    def __init__(self, embed_dim: int, feat_dim: int):
        super().__init__()
        self.gamma = nn.Sequential(nn.Linear(embed_dim, feat_dim), nn.Sigmoid())
        self.beta = nn.Linear(embed_dim, feat_dim)

    def forward(self, feat, embed):
        gamma = self.gamma(embed).unsqueeze(-1).unsqueeze(-1) + 1.0
        beta = self.beta(embed).unsqueeze(-1).unsqueeze(-1)
        return gamma * feat + beta


class PscSEDecoderStage(nn.Module):
    """Conv, P-scSE, Conv. The top stage uses the shortened P-scSE."""

    def __init__(self, in_ch: int, out_ch: int, r: int = 16, shortened: bool = False):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
        )
        self.pscse = ParallelScSE(out_ch, r, shortened)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(True),
        )

    def forward(self, x):
        return self.conv2(self.pscse(self.conv1(x)))


class SegmentationDecoder(nn.Module):
    """Progressive P4 to P1 decoder with FiLM conditioning and P-scSE stages."""

    def __init__(self, cfg: WILLIEConfig):
        super().__init__()
        self.cfg = cfg
        dim = cfg.fpn_dim
        self.films = nn.ModuleList(
            [FiLMConditioner(cfg.cls_embed_dim, dim) for _ in range(cfg.fpn_levels)]
        )
        self.stages = nn.ModuleList(
            [
                PscSEDecoderStage(
                    dim * 2,
                    dim,
                    cfg.pscse_reduction,
                    shortened=(i == cfg.fpn_levels - 2),
                )
                for i in range(cfg.fpn_levels - 1)
            ]
        )
        self.head = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(dim // 2),
            nn.ReLU(True),
            nn.Conv2d(dim // 2, dim // 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(dim // 4),
            nn.ReLU(True),
            nn.Conv2d(dim // 4, cfg.seg_out_channels, 1),
        )

    def forward(self, pyr, wound_embed, target_size: Optional[int] = None):
        if target_size is None:
            target_size = self.cfg.img_size
        cond = [self.films[i](pyr[i], wound_embed) for i in range(self.cfg.fpn_levels)]
        x = cond[-1]
        for s in range(self.cfg.fpn_levels - 1):
            skip = cond[self.cfg.fpn_levels - 2 - s]
            x = F.interpolate(
                x, size=skip.shape[-2:], mode="bilinear", align_corners=False
            )
            x = self.stages[s](torch.cat([x, skip], 1))
        x = F.interpolate(x, size=target_size, mode="bilinear", align_corners=False)
        return self.head(x)


class DetectionDecoder(nn.Module):
    """Anchor-free detection head (kept so the released weights load strictly)."""

    def __init__(self, cfg: WILLIEConfig):
        super().__init__()
        dim = cfg.fpn_dim
        self.level_w = nn.Parameter(torch.ones(cfg.fpn_levels) / cfg.fpn_levels)
        self.shared = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1, bias=False),
            nn.GroupNorm(32, dim),
            nn.GELU(),
            nn.Conv2d(dim, dim, 3, padding=1, bias=False),
            nn.GroupNorm(32, dim),
            nn.GELU(),
        )
        self.obj = nn.Conv2d(dim, 1, 1)
        self.bbox = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(dim // 2, 4, 1),
            nn.Sigmoid(),
        )
        self.cls = nn.Conv2d(dim, cfg.num_classes, 1)

    def forward(self, pyr):
        weights = F.softmax(self.level_w, 0)
        target = pyr[0].shape[-2:]
        fused = sum(
            weights[i]
            * (
                F.interpolate(p, target, mode="bilinear", align_corners=False)
                if p.shape[-2:] != target
                else p
            )
            for i, p in enumerate(pyr)
        )
        feat = self.shared(fused)
        return {
            "objectness": self.obj(feat),
            "bbox": self.bbox(feat),
            "det_cls": self.cls(feat),
        }


# ──────────────────────────────────────────────────────────────────────────────
# Metrics used by the original evaluation (kept here so numbers are comparable)
# ──────────────────────────────────────────────────────────────────────────────


def mask_to_boxes(mask: np.ndarray, min_area: int = 50) -> List[List[float]]:
    """Connected components (8-neighbourhood) to normalised ``[x1, y1, x2, y2]`` boxes.

    Args:
        mask: ``(H, W)`` array; values above 0.5 are foreground.
        min_area: Components smaller than this many pixels are dropped.
    """
    from scipy import ndimage

    binary = mask > 0.5
    if not binary.any():
        return []
    height, width = binary.shape
    labeled, count = ndimage.label(binary, structure=np.ones((3, 3), dtype=int))
    boxes = []
    for i, sl in enumerate(ndimage.find_objects(labeled), start=1):
        if sl is None:
            continue
        area = int((labeled[sl] == i).sum())
        if area < min_area:
            continue
        y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        boxes.append([x0 / width, y0 / height, x1 / width, y1 / height])
    return boxes


def _iou(a: List[float], b: List[float]) -> float:
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / (union + 1e-8)


def ap50_single(
    pred_boxes: List[List[float]],
    gt_boxes: List[List[float]],
    iou_thresh: float = 0.5,
) -> float:
    """Per-image AP@0.5 as the original notebook computes it.

    This is ``precision * recall`` at a single IoU threshold with every
    predicted box treated as equally confident; it is an approximation of
    the COCO AP, not the COCO AP itself.
    """
    if not gt_boxes and not pred_boxes:
        return 1.0
    if not gt_boxes or not pred_boxes:
        return 0.0
    matched, tp, fp = set(), 0, 0
    for pb in pred_boxes:
        best_iou, best_j = 0.0, -1
        for j, gb in enumerate(gt_boxes):
            if j in matched:
                continue
            iou = _iou(pb, gb)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh and best_j >= 0:
            tp += 1
            matched.add(best_j)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return precision * recall if (precision + recall) > 0 else 0.0


@torch.no_grad()
def dice_score(
    seg_logit: torch.Tensor, mask: torch.Tensor, threshold: float = 0.5
) -> List[float]:
    """Per-image Dice between ``sigmoid(seg_logit) > threshold`` and ``mask > 0.5``."""
    pred = (torch.sigmoid(seg_logit) > threshold).float().flatten(1)
    gt = (mask.to(pred.device) > 0.5).float().flatten(1)
    scores = []
    for p, g in zip(pred, gt):
        union = p.sum() + g.sum()
        if union == 0:
            scores.append(1.0)
        else:
            scores.append(float(2 * (p * g).sum() / (union + 1e-8)))
    return scores


@torch.no_grad()
def localization_ap50(
    seg_logit: torch.Tensor,
    mask: torch.Tensor,
    threshold: float = 0.5,
    min_area: int = 50,
) -> List[float]:
    """Per-image AP@0.5 of boxes derived from the predicted and true masks."""
    pred = (torch.sigmoid(seg_logit) > threshold).float().cpu().numpy()[:, 0]
    gt = mask.float().cpu().numpy()[:, 0]
    return [
        ap50_single(mask_to_boxes(p, min_area), mask_to_boxes(g, min_area))
        for p, g in zip(pred, gt)
    ]


# ──────────────────────────────────────────────────────────────────────────────
# PyHealth model
# ──────────────────────────────────────────────────────────────────────────────


def _dice_loss(pred: torch.Tensor, target: torch.Tensor, smooth: float = 1.0):
    p = torch.sigmoid(pred).flatten(1)
    t = target.flatten(1)
    inter = (p * t).sum(1)
    return 1.0 - ((2 * inter + smooth) / (p.sum(1) + t.sum(1) + smooth)).mean()


class WILLIEMini(BaseModel):
    """WILLIE-MINI: DINOv2 + FPN + WA-CSA with classification, segmentation
    and localization heads.

    The classification head is trained with the dataset's multiclass label
    (task :class:`~pyhealth.tasks.WILLIEClassification`); the segmentation
    head with the ``"mask"`` output of
    :class:`~pyhealth.tasks.WILLIESegmentation`. Localization boxes are
    derived from the predicted mask (:meth:`predict_boxes`), as in the
    original evaluation.

    Compared with the original multi-task training loop, each PyHealth
    dataset trains one task, so the learned uncertainty weighting between
    tasks is not used: the loss is the sum of the terms present in the batch
    (cross-entropy with label smoothing, BCE + Dice, detection objectness)
    plus ``aux_weight`` times the router load-balancing loss.

    Args:
        dataset: A :class:`~pyhealth.datasets.SampleDataset` produced by one of
            the WILLIE tasks, or ``None`` to build the bare network.
        pretrained_backbone: Download DINOv2 weights through ``torch.hub``.
        backbone: Optional ready-made backbone module (used in tests).
        config: Optional :class:`WILLIEConfig`; ``num_classes`` is taken from
            the dataset's label processor when there is one.
        img_size: Input resolution (ignored when ``config`` is given).
        seg_size: Output resolution of the segmentation logits.
        label_smoothing: Label smoothing of the classification loss.
        aux_weight: Weight of the router load-balancing loss.

    Examples:
        >>> from pyhealth.datasets import WILLIEDataset, get_dataloader
        >>> from pyhealth.tasks import WILLIEClassification
        >>> dataset = WILLIEDataset(root="/content/Willie")
        >>> samples = dataset.set_task(WILLIEClassification(split="test"))
        >>> model = WILLIEMini(samples, pretrained_backbone=False)
        >>> model.load_original_checkpoint("willie_mini_fold0_best.pt")
    """

    def __init__(
        self,
        dataset=None,
        pretrained_backbone: bool = True,
        backbone: Optional[nn.Module] = None,
        config: Optional[WILLIEConfig] = None,
        img_size: int = 518,
        seg_size: int = 512,
        label_smoothing: float = 0.1,
        aux_weight: float = 0.01,
    ):
        super().__init__(dataset=dataset)
        cfg = config if config is not None else WILLIEConfig(img_size=img_size)

        processors = self.label_processors if dataset is not None else {}
        self.cls_key: Optional[str] = next(iter(processors), None)
        if self.cls_key is not None:
            cfg = replace(cfg, num_classes=processors[self.cls_key].output_size())
        self.mask_key: Optional[str] = (
            "mask" if "mask" in self.label_keys else None
        )
        self.image_key: str = (
            "image" if "image" in self.feature_keys or not self.feature_keys
            else self.feature_keys[0]
        )

        self.cfg = cfg
        self.seg_size = seg_size
        self.label_smoothing = label_smoothing
        self.aux_weight = aux_weight

        self.encoder = DINOv2MultiScale(cfg, pretrained_backbone, backbone)
        self.fpn = FeaturePyramidNeck(cfg)
        self.wa_csa = WA_CSA_Stack(cfg)
        self.cls_decoder = ClassificationDecoder(cfg)
        self.seg_decoder = SegmentationDecoder(cfg)
        self.det_decoder = DetectionDecoder(cfg)

    # ── backbone freezing (the original recipe freezes it for 5 epochs) ──
    def freeze_backbone(self) -> None:
        """Freezes the DINOv2 backbone."""
        self.encoder.set_frozen(True)

    def unfreeze_backbone(self) -> None:
        """Unfreezes the DINOv2 backbone."""
        self.encoder.set_frozen(False)

    def backbone_parameters(self):
        """Parameters of the DINOv2 backbone (for a lower learning rate)."""
        return self.encoder.parameters()

    def head_parameters(self):
        """All parameters outside the backbone."""
        return [p for n, p in self.named_parameters() if not n.startswith("encoder.")]

    # ── forward ──
    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        """Forward pass.

        Args:
            **kwargs: Batch from a PyHealth dataloader. ``image`` is required;
                the multiclass label (classification task) and ``mask``
                (segmentation task) are optional and enable the losses.
                Pass ``return_seg=True`` to get the mask logits without a
                mask in the batch.

        Returns:
            ``logit``/``y_prob`` (classification), ``wound_embed``, and, when
            segmentation runs, ``seg_logit``, ``det_objectness``,
            ``det_bbox``, ``det_cls``. With labels or masks also ``loss`` and
            (for a class label) ``y_true``.
        """
        x = kwargs[self.image_key].to(self.device)
        pyr = self.wa_csa(self.fpn(self.encoder(x)))
        logits, embed = self.cls_decoder(pyr)

        if self.cls_key is not None:
            y_prob = self.label_processors[self.cls_key].activation(logits)
        else:
            y_prob = F.softmax(logits, dim=-1)
        out: Dict[str, torch.Tensor] = {
            "logit": logits,
            "y_prob": y_prob,
            "wound_embed": embed,
        }

        loss = None
        if self.cls_key is not None and self.cls_key in kwargs:
            y_true = kwargs[self.cls_key].to(self.device).long()
            out["y_true"] = y_true
            loss = F.cross_entropy(
                logits, y_true, label_smoothing=self.label_smoothing
            )

        has_mask = self.mask_key is not None and self.mask_key in kwargs
        if has_mask or kwargs.get("return_seg", False):
            seg = self.seg_decoder(pyr, embed, self.seg_size)
            det = self.det_decoder(pyr)
            out["seg_logit"] = seg
            out["det_objectness"] = det["objectness"]
            out["det_bbox"] = det["bbox"]
            out["det_cls"] = det["det_cls"]
            if has_mask:
                target = kwargs[self.mask_key].to(self.device).float()
                if target.shape[-2:] != seg.shape[-2:]:
                    target = F.interpolate(
                        target, seg.shape[-2:], mode="bilinear", align_corners=False
                    )
                out["y_true"] = target
                seg_loss = F.binary_cross_entropy_with_logits(
                    seg, target
                ) + _dice_loss(seg, target)
                obj = det["objectness"]
                obj_target = F.interpolate(
                    target, obj.shape[-2:], mode="bilinear", align_corners=False
                )
                det_loss = F.binary_cross_entropy_with_logits(
                    obj, (obj_target > 0.3).float()
                )
                seg_total = seg_loss + det_loss
                loss = seg_total if loss is None else loss + seg_total

        if loss is not None:
            out["loss"] = loss + self.aux_weight * self.cls_decoder.router.aux_loss
        return out

    # ── helpers ──
    @torch.no_grad()
    def predict_boxes(
        self, seg_logit: torch.Tensor, threshold: float = 0.5, min_area: int = 50
    ) -> List[List[List[float]]]:
        """Boxes (normalised ``[x1, y1, x2, y2]``) from predicted masks."""
        masks = (torch.sigmoid(seg_logit) > threshold).float().cpu().numpy()[:, 0]
        return [mask_to_boxes(m, min_area) for m in masks]

    def load_original_checkpoint(
        self, path: str, map_location: str = "cpu"
    ) -> Dict[str, Any]:
        """Loads a checkpoint released with the original WILLIE code.

        The checkpoint is a pickle written by the original training code (it
        holds numpy scalars in its metrics), so it is loaded with
        ``weights_only=False``; only use files from a source you trust.

        Raises:
            RuntimeError: If any weight is missing or unexpected, apart from
                the PyHealth-only ``_dummy_param``.

        Returns:
            The checkpoint dictionary (``config``, ``metrics``, ``epoch`` ...).
        """
        ckpt = torch.load(path, map_location=map_location, weights_only=False)
        state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
        result = self.load_state_dict(state, strict=False)
        missing = [k for k in result.missing_keys if k != "_dummy_param"]
        if missing or result.unexpected_keys:
            raise RuntimeError(
                f"Checkpoint does not match the model. Missing: {missing[:5]}, "
                f"unexpected: {list(result.unexpected_keys)[:5]}"
            )
        return ckpt

In [ ]:
%%writefile /content/PyHealth/tests/core/test_willie.py
"""Tests for the WILLIE dataset, tasks, processors and WILLIE-MINI model.

All tests use synthetic images and a tiny stand-in backbone, so they need no
downloads. The equivalence test also needs a checkout of
https://github.com/Qian-Group-HRI/Willie (set ``WILLIE_REPO``); it is skipped
otherwise.

Author: <your name> (<your NetID>)
"""

import json
import os
import tempfile
import unittest
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pyhealth.datasets import WILLIEDataset, get_dataloader
from pyhealth.models import WILLIEConfig, WILLIEMini
from pyhealth.models.willie_mini import (
    ap50_single,
    dice_score,
    mask_to_boxes,
)
from pyhealth.processors import WoundImageProcessor, WoundMaskProcessor
from pyhealth.tasks import WILLIEClassification, WILLIESegmentation

IMG = 56  # 4 x 4 patch grid
CLASSES = ["diabetic", "pressure", "surgical", "venous", "no_wound"]


class _Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)

    def forward(self, x):
        return x + self.fc(x)


class TinyViT(nn.Module):
    """Stand-in for DINOv2: 12 blocks, a CLS token and 14-pixel patches."""

    def __init__(self, dim=32, depth=12):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, dim, 14, 14)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.blocks = nn.ModuleList([_Block(dim) for _ in range(depth)])

    def forward(self, x):
        tok = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(len(tok), -1, -1), tok], 1)
        for block in self.blocks:
            x = block(x)
        return x[:, 0]


def tiny_config():
    return WILLIEConfig(
        backbone_dim=32,
        img_size=IMG,
        fpn_dim=32,
        cls_embed_dim=16,
        wa_csa_heads=8,
    )


def make_tree(root: Path, n_cls=20, n_seg=8):
    """Writes a miniature copy of the WILLIE repo layout with fake images."""
    rng = np.random.RandomState(0)
    man = root / "Manifests_and_Splits" / "manifests"
    idx = root / "data" / "processed" / "index"
    man.mkdir(parents=True)
    idx.mkdir(parents=True)

    def img(rel):
        p = root / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(p), rng.randint(0, 255, (40, 30, 3), dtype=np.uint8))

    rows = []
    for i in range(n_cls):
        rel = f"data/AZH/train/c{i % 5}/{i}.png"
        img(rel)
        rows.append(
            dict(
                image_path=rel,
                original_class=CLASSES[i % 5],
                unified_label=i % 5,
                unified_class=CLASSES[i % 5],
                source="AZH-azh_train",
                azh_split="train",
            )
        )
    df = pd.DataFrame(rows)
    df.iloc[:12].to_csv(man / "cls_train.csv", index=False)
    df.iloc[12:16].to_csv(man / "cls_val.csv", index=False)
    test = df.iloc[16:].copy()
    test["source"] = "AZH-azh_test"
    test.to_csv(man / "cls_test.csv", index=False)

    seg = []
    for i in range(n_seg):
        split = "train" if i < 5 else "val"
        ip, mp = f"data/FUSeg/{split}/images/{i}.png", f"data/FUSeg/{split}/labels/{i}.png"
        img(ip)
        m = np.zeros((40, 30), np.uint8)
        m[8:20, 6:18] = 255
        (root / mp).parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(root / mp), m)
        seg.append(dict(sample_id=f"s{i}", dataset="FUSeg", split=split, task="seg",
                        image_path=ip, mask_path=mp, is_labeled=1))
    ti = "data/FUSeg/test/images/99.png"
    img(ti)
    seg.append(dict(sample_id="s99", dataset="FUSeg", split="test", task="seg",
                    image_path=ti, mask_path=np.nan, is_labeled=0))
    cls_idx = [dict(sample_id="c", dataset="AZH", split="train", task="cls",
                    image_path=rows[0]["image_path"], mask_path=np.nan, is_labeled=1)]
    pd.DataFrame(seg + cls_idx).to_csv(idx / "data_index.csv", index=False)


def make_folds(path: Path, n_trainval=16, n_seg=8):
    folds = {}
    for f in range(4):
        va = [i for i in range(n_trainval) if i % 4 == f]
        tr = [i for i in range(n_trainval) if i % 4 != f]
        sva = [i for i in range(n_seg) if i % 4 == f]
        folds[f] = dict(cls_tr=tr, cls_va=va,
                        seg_tr=[i for i in range(n_seg) if i % 4 != f], seg_va=sva)
    torch.save({"n_folds": 4, "folds": folds}, path)


class TestWILLIE(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.tmp = tempfile.TemporaryDirectory()
        cls.root = Path(cls.tmp.name) / "Willie"
        cls.root.mkdir()
        make_tree(cls.root)
        cls.folds = Path(cls.tmp.name) / "folds.pt"
        make_folds(cls.folds)
        cls.dataset = WILLIEDataset(
            root=str(cls.root),
            folds_path=str(cls.folds),
            cache_dir=str(Path(cls.tmp.name) / "cache"),
        )
        cls.cls_task = WILLIEClassification(split="train", image_size=IMG)
        cls.cls_samples = cls.dataset.set_task(cls.cls_task)
        cls.seg_samples = cls.dataset.set_task(
            WILLIESegmentation(split="train", image_size=IMG, mask_size=IMG)
        )

    @classmethod
    def tearDownClass(cls):
        cls.cls_samples.close()
        cls.seg_samples.close()
        cls.tmp.cleanup()

    # ── metadata ──
    def test_metadata(self):
        df = pd.read_csv(self.root / "willie-metadata-pyhealth.csv")
        self.assertEqual((df.kind == "cls").sum(), 20)
        self.assertEqual((df.kind == "seg").sum(), 8)  # unlabeled FUSeg test is dropped
        self.assertTrue(all(os.path.isabs(p) for p in df.image))
        self.assertTrue(all(os.path.isfile(p) for p in df.image))
        trainval = df[(df.kind == "cls") & (df.split != "test")]
        self.assertEqual(sorted(trainval.cv_fold.unique()), [0, 1, 2, 3])
        self.assertTrue((df[df.split == "test"].cv_fold == -1).all())

    def test_mismatching_folds_raise(self):
        bad = Path(self.tmp.name) / "bad.pt"
        make_folds(bad, n_trainval=40)
        with self.assertRaises(ValueError):
            WILLIEDataset.prepare_metadata(str(self.root), str(bad))
        WILLIEDataset.prepare_metadata(str(self.root), str(self.folds))  # restore

    def test_missing_image_raises(self):
        victim = next((self.root / "data" / "AZH").rglob("0.png"))
        data = victim.read_bytes()
        victim.unlink()
        try:
            with self.assertRaises(FileNotFoundError):
                WILLIEDataset.prepare_metadata(str(self.root))
        finally:
            victim.write_bytes(data)
            WILLIEDataset.prepare_metadata(str(self.root), str(self.folds))

    # ── tasks ──
    def test_classification_samples(self):
        self.assertEqual(len(self.cls_samples), 12)
        sample = self.cls_samples[0]
        self.assertEqual(tuple(sample["image"].shape), (3, IMG, IMG))
        self.assertTrue(os.path.isfile(sample["image_path"]))  # kept for alignment with logits
        labels = sorted(int(self.cls_samples[i]["label"]) for i in range(12))
        self.assertEqual(labels, sorted(i % 5 for i in range(12)))  # ids kept as given

    def test_label_space_is_fixed_to_five_classes(self):
        proc = self.cls_samples.output_processors["label"]
        self.assertEqual(proc.output_size(), 5)

    def test_split_and_fold_selection(self):
        test = self.dataset.set_task(WILLIEClassification(split="test", image_size=IMG))
        self.assertEqual(len(test), 4)
        test.close()
        tr = self.dataset.set_task(WILLIEClassification(fold=1, fold_role="train", image_size=IMG))
        va = self.dataset.set_task(WILLIEClassification(fold=1, fold_role="val", image_size=IMG))
        self.assertEqual((len(tr), len(va)), (12, 4))
        tr.close()
        va.close()

    def test_task_argument_validation(self):
        with self.assertRaises(ValueError):
            WILLIEClassification(fold=1)
        with self.assertRaises(ValueError):
            WILLIEClassification(split="train", fold=1, fold_role="val")
        with self.assertRaises(ValueError):
            WILLIEClassification(fold=1, fold_role="test")

    def test_segmentation_samples(self):
        self.assertEqual(len(self.seg_samples), 5)
        mask = self.seg_samples[0]["mask"]
        self.assertEqual(tuple(mask.shape), (1, IMG, IMG))
        self.assertEqual(set(mask.unique().tolist()), {0.0, 1.0})

    # ── processors ──
    def test_image_processor_matches_original_recipe(self):
        path = next((self.root / "data" / "AZH").rglob("1.png"))
        out = WoundImageProcessor(image_size=IMG).process(path)
        ref = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
        ref = cv2.resize(ref, (IMG, IMG), interpolation=cv2.INTER_LINEAR)
        mean = np.array([0.485, 0.456, 0.406], np.float32)
        std = np.array([0.229, 0.224, 0.225], np.float32)
        ref = ((ref.astype(np.float32) / 255 - mean) / std).transpose(2, 0, 1)
        np.testing.assert_allclose(out.numpy(), ref, atol=1e-6)

    def test_mask_processor_is_binary_and_nearest(self):
        path = next((self.root / "data" / "FUSeg" / "train" / "labels").glob("0.png"))
        out = WoundMaskProcessor(mask_size=100).process(path)
        self.assertEqual(set(out.unique().tolist()), {0.0, 1.0})
        with self.assertRaises(FileNotFoundError):
            WoundMaskProcessor().process(self.root / "nope.png")

    # ── model ──
    def _model(self, samples, **kw):
        return WILLIEMini(samples, backbone=TinyViT(), config=tiny_config(), seg_size=IMG, **kw)

    def test_classification_forward_backward(self):
        model = self._model(self.cls_samples)
        batch = next(iter(get_dataloader(self.cls_samples, batch_size=4)))
        out = model(**batch)
        self.assertEqual(tuple(out["logit"].shape), (4, 5))
        self.assertTrue(torch.allclose(out["y_prob"].sum(1), torch.ones(4), atol=1e-5))
        out["loss"].backward()
        # embed_head only feeds the segmentation decoder, so it has no gradient here
        no_grad = [n for n, p in model.cls_decoder.named_parameters() if p.grad is None]
        self.assertTrue(all(n.startswith("embed_head") for n in no_grad), no_grad)
        self.assertTrue(all(p.grad is None for p in model.encoder.parameters()))  # frozen

    def test_segmentation_forward_backward_and_boxes(self):
        model = self._model(self.seg_samples)
        batch = next(iter(get_dataloader(self.seg_samples, batch_size=2)))
        out = model(**batch)
        self.assertEqual(tuple(out["seg_logit"].shape), (2, 1, IMG, IMG))
        self.assertIn("loss", out)
        out["loss"].backward()
        self.assertTrue(all(p.grad is not None for p in model.cls_decoder.embed_head.parameters()))
        boxes = model.predict_boxes(out["seg_logit"].detach())
        self.assertEqual(len(boxes), 2)

    def test_unfreeze(self):
        model = self._model(self.cls_samples)
        model.unfreeze_backbone()
        self.assertTrue(all(p.requires_grad for p in model.encoder.parameters()))
        model.freeze_backbone()
        self.assertFalse(any(p.requires_grad for p in model.encoder.parameters()))

    def test_load_checkpoint_rejects_mismatch(self):
        model = self._model(self.cls_samples)
        path = Path(self.tmp.name) / "bad_ckpt.pt"
        torch.save({"model_state_dict": {"foo": torch.zeros(1)}}, path)
        with self.assertRaises(RuntimeError):
            model.load_original_checkpoint(str(path))

    def test_load_checkpoint_roundtrip(self):
        a, b = self._model(self.cls_samples), self._model(self.cls_samples)
        path = Path(self.tmp.name) / "ok_ckpt.pt"
        torch.save({"model_state_dict": a.state_dict(), "epoch": 3}, path)
        self.assertEqual(b.load_original_checkpoint(str(path))["epoch"], 3)
        for (_, x), (_, y) in zip(a.state_dict().items(), b.state_dict().items()):
            self.assertTrue(torch.equal(x, y))


class TestMetrics(unittest.TestCase):
    def test_dice(self):
        logit = torch.full((2, 1, 8, 8), -10.0)
        logit[0, 0, :4] = 10.0
        mask = torch.zeros(2, 1, 8, 8)
        mask[0, 0, :4] = 1.0
        self.assertEqual(dice_score(logit, mask), [1.0, 1.0])  # perfect and empty-empty
        mask[0, 0, :2] = 0.0
        self.assertAlmostEqual(dice_score(logit, mask)[0], 2 * 16 / (32 + 16), places=5)

    def test_ap50_cases(self):
        box = [0.1, 0.1, 0.5, 0.5]
        self.assertEqual(ap50_single([], []), 1.0)
        self.assertEqual(ap50_single([box], []), 0.0)
        self.assertEqual(ap50_single([], [box]), 0.0)
        self.assertAlmostEqual(ap50_single([box], [box]), 1.0, places=5)
        self.assertEqual(ap50_single([[0.6, 0.6, 0.9, 0.9]], [box]), 0.0)

    def test_boxes_match_opencv_components(self):
        rng = np.random.RandomState(1)
        for _ in range(20):
            mask = (rng.rand(64, 64) > 0.93).astype(np.float32)
            mask = cv2.dilate(mask, np.ones((3, 3)))
            n, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
            ref = sorted(
                [x / 64, y / 64, (x + w) / 64, (y + h) / 64]
                for x, y, w, h, a in stats[1:]
                if a >= 50
            )
            self.assertEqual(sorted(mask_to_boxes(mask, 50)), ref)


@unittest.skipUnless(
    os.environ.get("WILLIE_REPO") and Path(os.environ.get("WILLIE_REPO", "")).exists(),
    "set WILLIE_REPO to a checkout of Qian-Group-HRI/Willie",
)
class TestEquivalenceWithOriginal(unittest.TestCase):
    """The PyHealth model must be the original network, weight for weight."""

    def test_same_outputs_as_original_code(self):
        nb = json.load(open(Path(os.environ["WILLIE_REPO"]) / "06_WILLIE_MINI.ipynb"))
        src = "".join(nb["cells"][0]["source"])
        body = src[src.index("# 2. MODEL CONFIGURATIONS"): src.index("# 12. CHECKPOINT MANAGER")]
        ns = {}
        exec(
            "import torch, torch.nn as nn, torch.nn.functional as F\n"
            "from dataclasses import dataclass, field\n"
            "from typing import List, Optional, Dict, Tuple, Any\n"
            "from einops import rearrange\n" + body,
            ns,
        )
        cfg = tiny_config()
        ocfg = ns["ModelConfig"](
            backbone_dim=cfg.backbone_dim, img_size=cfg.img_size, fpn_dim=cfg.fpn_dim,
            cls_embed_dim=cfg.cls_embed_dim, wa_csa_heads=cfg.wa_csa_heads,
        )
        real_load = torch.hub.load
        torch.hub.load = lambda *a, **k: TinyViT()
        try:
            torch.manual_seed(0)
            original = ns["WILLIEModel"](ocfg).eval()
        finally:
            torch.hub.load = real_load
        mine = WILLIEMini(None, backbone=TinyViT(), config=cfg, seg_size=IMG).eval()

        result = mine.load_state_dict(original.state_dict(), strict=False)
        self.assertEqual([k for k in result.missing_keys if k != "_dummy_param"], [])
        self.assertEqual(list(result.unexpected_keys), [])

        x = torch.randn(3, 3, IMG, IMG)
        with torch.no_grad():
            ref = original(x, target_seg_size=IMG)
            out = mine(image=x, return_seg=True)
        torch.testing.assert_close(out["logit"], ref["cls_logits"], atol=1e-5, rtol=1e-5)
        torch.testing.assert_close(out["seg_logit"], ref["seg_mask"], atol=1e-5, rtol=1e-5)
        torch.testing.assert_close(out["det_objectness"], ref["det_objectness"], atol=1e-5, rtol=1e-5)


if __name__ == "__main__":
    unittest.main()

In [ ]:
# Register the new classes: append one line per __init__.py (skipped if already present)
from pathlib import Path
REGISTER = {
    "tasks/__init__.py":      "from .willie_tasks import WILLIEClassification, WILLIESegmentation",
    "processors/__init__.py": "from .wound_processors import WoundImageProcessor, WoundMaskProcessor",
    "datasets/__init__.py":   "from .willie import WILLIEDataset",
    "models/__init__.py":     "from .willie_mini import WILLIEMini, WILLIEConfig",
}
for rel, line in REGISTER.items():
    f = PYHEALTH_DIR / "pyhealth" / rel
    text = f.read_text()
    if line not in text:
        f.write_text(text.rstrip("\n") + "\n" + line + "\n")
        print("added  ", rel)
    else:
        print("exists ", rel)

added   tasks/__init__.py
added   processors/__init__.py
added   datasets/__init__.py
added   models/__init__.py


**Restart the runtime now** (`Runtime → Restart session`). If PyHealth was imported earlier in this session, the new classes are not visible. Files under `/content` survive the restart but variables do not, so re-run the path-configuration cell of §1 (only that cell; the downloads need not be repeated) before continuing.

In [ ]:
# Unit tests: synthetic images and a tiny stand-in backbone (no downloads).
# With WILLIE_REPO set, the test that compares outputs against the original code also runs.
!cd {PYHEALTH_DIR} && WILLIE_REPO={REPO_DIR} python -m pytest tests/core/test_willie.py -q -p no:cacheprovider --show-capture=no

19 passed, 1 warning in 44.12s


What the tests cover: metadata generation and error handling (missing files, mismatched fold file); a fixed label space (5 classes, integer labels preserved); split / fold selection; preprocessing identical to the original recipe; binary masks; classification and segmentation forward/backward passes; backbone freezing; checkpoint-loading validation; Dice / AP@0.5 / box extraction against OpenCV; and — most importantly — that the original `WILLIEModel` weights load into `WILLIEMini` and produce the same logits, mask logits and objectness maps.

## 4. PyHealth dataset
`WILLIEDataset` writes a metadata CSV (`willie-metadata-pyhealth.csv`) at the repo root and attaches the public 5-fold indices as a `cv_fold` column. It raises a `ValueError` if the fold file does not match the manifest row counts.

In [ ]:
import pandas as pd
from pyhealth.datasets import WILLIEDataset

dataset = WILLIEDataset(root=str(REPO_DIR), folds_path=str(FOLDS_PT), cache_dir="/content/pyhealth_cache")
meta = pd.read_csv(REPO_DIR / "willie-metadata-pyhealth.csv")
print(meta.groupby(["kind", "split"]).size())
print(meta[meta.kind == "cls"].groupby(["split", "cv_fold"]).size().unstack(fill_value=0))

kind  split
cls   test     234
      train    918
      val      162
seg   train    610
      val      400
dtype: int64
cv_fold   -1    0    1    2    3    4
split                                
test     234    0    0    0    0    0
train      0  190  178  182  179  189
val        0   26   38   34   37   27


Classification: train 918 / val 162 / test 234 (1,314 in total); segmentation: FUSeg train 610 / val 400 (the 200 FUSeg test images have no public masks and are excluded). `cls_train + cls_val = 1,080` equals fold 0 of `5fold_splits_v2.pt` (864 train + 216 val), and the per-fold counts above sum to the same total.

## 5. Verification against the public weights
The same checkpoint is loaded into `WILLIEMini` to check (1) that its logits equal the reference logits from §2, and (2) that the public fold indices match the manifests (the checkpoint records 84.72% on fold 0 validation, i.e. 183/216). The helper functions for evaluation and training are defined first.

In [ ]:
import time
import numpy as np
import torch
from pyhealth.datasets import get_dataloader
from pyhealth.models.willie_mini import dice_score, localization_ap50

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def evaluate_cls(model, dataset, batch_size=8):
    """Returns (accuracy %, logits, image_paths) over a classification SampleDataset."""
    model.eval()
    logits, labels, paths = [], [], []
    for batch in get_dataloader(dataset, batch_size=batch_size, shuffle=False):
        out = model(**batch)
        logits.append(out["logit"].float().cpu())
        labels.append(out["y_true"].cpu())
        paths += list(batch["image_path"])
    logits, labels = torch.cat(logits), torch.cat(labels)
    acc = (logits.argmax(1) == labels).float().mean().item() * 100
    return acc, logits, paths


@torch.no_grad()
def evaluate_seg(model, dataset, batch_size=4):
    """Returns (mean Dice %, mean mask-derived AP@0.5 %) over a segmentation SampleDataset."""
    model.eval()
    dice, ap = [], []
    for batch in get_dataloader(dataset, batch_size=batch_size, shuffle=False):
        # return_seg=True yields mask logits even if the model was built from a classification dataset
        out = model(**batch, return_seg=True)
        target = batch["mask"].to(out["seg_logit"].device)
        dice += dice_score(out["seg_logit"], target)
        ap += localization_ap50(out["seg_logit"], target)
    return float(np.mean(dice) * 100), float(np.mean(ap) * 100)


def train_cls(model, train_ds, val_ds, epochs=50, freeze_epochs=5, batch_size=4,
              accum=2, lr_head=1e-4, lr_backbone=1e-5, weight_decay=1e-4,
              patience=12, grad_clip=1.0, ckpt_path=None):
    """Classification-only training with the original WILLIE-MINI recipe:
    backbone frozen for `freeze_epochs`, AdamW with two learning rates, cosine
    warm restarts, mixed precision on GPU, gradient accumulation, early stopping.
    Differences from the original: no augmentation, no class-balanced sampler,
    no segmentation/detection loss."""
    model.to(DEV)
    opt = torch.optim.AdamW(
        [{"params": model.head_parameters(), "lr": lr_head},
         {"params": list(model.backbone_parameters()), "lr": lr_backbone}],
        weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2, eta_min=1e-7)
    use_amp = DEV.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    model.freeze_backbone()
    best, bad, history = -1.0, 0, []
    for epoch in range(epochs):
        if epoch == freeze_epochs:
            model.unfreeze_backbone()
        model.train()
        t0, total, steps = time.time(), 0.0, 0
        loader = get_dataloader(train_ds, batch_size=batch_size, shuffle=True, seed=epoch, drop_last=True)
        opt.zero_grad()
        for step, batch in enumerate(loader):
            with torch.autocast(DEV.type, dtype=torch.float16, enabled=use_amp):
                loss = model(**batch)["loss"] / accum
            scaler.scale(loss).backward()
            if (step + 1) % accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(opt); scaler.update(); opt.zero_grad()
            total += loss.item() * accum; steps += 1
        sched.step()
        acc, _, _ = evaluate_cls(model, val_ds)
        history.append({"epoch": epoch, "train_loss": total / max(steps, 1), "val_acc": acc})
        print(f"epoch {epoch:02d} loss={total / max(steps, 1):.3f} val_acc={acc:.2f}% ({time.time() - t0:.0f}s)")
        if acc > best:
            best, bad = acc, 0
            if ckpt_path:
                torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_acc": acc}, ckpt_path)
        else:
            bad += 1
            if bad >= patience:
                print("early stop"); break
    return best, history

In [ ]:
import os, torch
from pyhealth.models import WILLIEMini
from pyhealth.tasks import WILLIEClassification, WILLIESegmentation

# (1) cls_test: PyHealth pipeline vs. the reference logits from section 2
test_ds = dataset.set_task(WILLIEClassification(split="test"))
model = WILLIEMini(test_ds, pretrained_backbone=False)       # backbone weights come from the checkpoint
model.load_original_checkpoint(ckpt_path)                    # raises RuntimeError if any key differs
model.to(DEV)

acc, logits, paths = evaluate_cls(model, test_ds)
ref = torch.load(REF_LOGITS, weights_only=True)
pos = {p: i for i, p in enumerate(ref["paths"])}
order = [pos[os.path.relpath(p, REPO_DIR)] for p in paths]
ref_logits = ref["logits"][order]
print(f"PyHealth cls_test acc = {acc:.2f}%   (original code path: 87.61%)")
print("max |logit diff|     =", (logits - ref_logits).abs().max().item())
print("argmax agreement    =", (logits.argmax(1) == ref_logits.argmax(1)).float().mean().item())

# (2) public fold-0 validation indices: the checkpoint records 84.72% (= 183/216)
val_ds = dataset.set_task(WILLIEClassification(fold=0, fold_role="val"))
acc_v, _, _ = evaluate_cls(model, val_ds)
print(f"fold 0 val: n={len(val_ds)}  acc={acc_v:.2f}%   (checkpoint record: 84.72%)")

PyHealth cls_test acc = 87.61%   (original code path: 87.61%)
max |logit diff|     = 0.0
argmax agreement    = 1.0
fold 0 val: n=216  acc=84.72%   (checkpoint record: 84.72%)


**Findings**
- **Same function as the original.** The maximum logit difference is exactly 0.0 and all 234 predictions agree, so preprocessing, model port and label mapping are equivalent to the authors' code on real data, not only on the synthetic test.
- **Public splits reproduce.** Fold 0 validation (n = 216) gives 84.72%, the value stored in the checkpoint, so `5fold_splits_v2.pt` is consistent with the GitHub manifests for classification.

In [ ]:
# (3) FUSeg segmentation and mask-derived box AP@0.5 on fold 0 validation
seg_val = dataset.set_task(WILLIESegmentation(fold=0, fold_role="val"))
dice, ap50 = evaluate_seg(model, seg_val)
print(f"fold 0 seg val: n={len(seg_val)}  Dice={dice:.2f}%  AP@0.5(mask-box)={ap50:.2f}%")
print("checkpoint record: Dice 79.18, AP@0.5 81.66")

fold 0 seg val: n=202  Dice=79.47%  AP@0.5(mask-box)=81.29%
checkpoint record: Dice 79.18, AP@0.5 81.66


The segmentation numbers agree within 0.4 points (Dice 79.47 vs 79.18; AP 81.29 vs 81.66). This is an **approximate** comparison, for three reasons: (a) the segmentation fold ordering is assumed to be the FUSeg train and val rows concatenated in index order, which could not be confirmed against the original; (b) the original resizes the mask to 518 and then to 512 with bilinear interpolation, whereas this code loads a binary mask with nearest-neighbour at 512; (c) the original "AP@0.5" is not COCO AP but precision × recall of connected-component boxes derived from the predicted and ground-truth masks (minimum area 50), which `localization_ap50` reproduces. These are weights evaluated on segmentation data, not a model trained for segmentation here.

## 6. Training in PyHealth (classification only)
Follows the original schedule — backbone frozen for the first epochs, then unfrozen; head LR 1e-4, backbone LR 1e-5; cosine warm restarts; AMP; batch 4 × 2 accumulation; early stopping patience 12 — but trains **classification only**. Differences from the original: (1) no data augmentation, (2) no class-balanced sampler, (3) no joint segmentation/detection loss. The result must therefore not be compared directly with the README's cross-validation numbers (Acc 86.4 ± 1.2). The cell below is a **2-epoch smoke test** that checks the pipeline end-to-end; the full run uses `epochs=50, freeze_epochs=5` (roughly 30–40 minutes on a T4) and was not run for this submission.

In [ ]:
FOLD = 0
tr_ds = dataset.set_task(WILLIEClassification(fold=FOLD, fold_role="train"))
va_ds = dataset.set_task(WILLIEClassification(fold=FOLD, fold_role="val"))
mini = WILLIEMini(tr_ds)          # downloads pretrained DINOv2 ViT-S/14 weights via torch.hub
best, history = train_cls(mini, tr_ds, va_ds, epochs=2, freeze_epochs=1,
                          ckpt_path=str(CKPT_DIR / f"willie_mini_pyhealth_fold{FOLD}.pt"))
# For the full run, change to epochs=50, freeze_epochs=5
print("best val acc:", best)

epoch 00 loss=1.192 val_acc=70.83% (53s)
epoch 01 loss=0.930 val_acc=71.76% (80s)
best val acc: 71.75925970077515


Validation accuracy rises from 70.8% to 71.8% over two epochs (loss 1.19 → 0.93), which confirms that data loading, the frozen/unfrozen backbone schedule, the loss and checkpointing work. It says nothing about final accuracy.

## 7. Differences from the original and limitations

| Item | Original | This implementation |
|---|---|---|
| Module names / structure | `WILLIEModel` | Identical (public weights load; outputs match exactly, §5) |
| Training objective | Classification + segmentation + detection jointly, uncertainty-weighted loss | One PyHealth task per dataset; the model sums the losses of the targets present in the batch |
| Detection | Detection head trained; evaluation approximated from masks → boxes | Detection head kept for weight compatibility; same mask-based evaluation (`localization_ap50`) |
| Augmentation / sampler | albumentations, class-balanced sampler | None |
| Mask resolution | 518, then 512 with bilinear interpolation | 512 with nearest-neighbour |
| 5-fold splits | Indices from `StratifiedKFold` (seed 42) | Public `5fold_splits_v2.pt`; classification indices verified (§5), segmentation order unverified |
| Headline numbers | 5-fold ensemble + TTA (Acc 91.88) | Not reproduced; single-fold evaluation only |
| Training | Full multi-task schedule | Classification-only, 2-epoch smoke test run; full run not performed |

**Inconsistencies in the public release** (the checkpoint and notebook code were treated as the source of truth)
- The README describes MINI as 224×224 with 8 experts; the checkpoint's saved config says **518×518 with 2 experts**.
- `environment.txt` is a placeholder ("PLACEHOLDER — NOT REAL VERSIONS"), and the training notebook's input paths (`artifacts/willie_v2/...`, columns `img`/`mask`) do not match the public repository layout.
- The "AP@0.5" metric is not standard COCO AP (see §5).
- The detection manifests contain 810 + 400 = 1,210 rows, whereas `5fold_splits_v2.pt` indexes 976 + 244 = 1,220. The cause is unknown; detection was therefore not evaluated.
- Checkpoint loading uses `weights_only=False` because the official file contains numpy scalars; this should not be used for files of unknown origin.

## AI-tool usage statement
<!-- Fill in according to the course policy. -->
- Tool(s) used:
- What they were used for:
- What I ran and verified myself: